# Entrenar "Oye Asia" (openWakeWord) en Colab

Para Colab gratuito con GPU T4 (Python 3.13): **Entorno de ejecución → Cambiar tipo → T4 GPU**, y luego **Ejecutar todo**.
La celda 3 pide subir `sintetico.zip` y `mi_voz.zip` (de `generar_wake.py` y `empaquetar_mi_voz.py`).

Basado en `automatic_model_training.ipynb` de openWakeWord, con los arreglos del entrenamiento del 2026-11-05
(ROADMAP de Asia, J3). El modelo hereda el uso no comercial de ACAV100M (CC BY-NC-SA 4.0): personal, no distribuible.

## 0. Configuración

In [ ]:
# Lo que se cambia de una versión a otra. El resto, como en custom_model.yml de openWakeWord.
MODEL_NAME = "oye_asia_v0.2"
TARGET_PHRASE = "oye Asia"
STEPS = 50000
AUGMENTATION_ROUNDS = 2
MAX_NEGATIVE_WEIGHT = 1500
TARGET_FP_PER_HOUR = 0.2
LAYER_SIZE = 32

# Datos
TEST_SHARE_SYNTHETIC = 0.10   # de los clips de Piper, a prueba
TEST_SHARE_MINE = 0.15        # de los clips del usuario, a prueba
MINE_REPEAT = 10              # cada clip del usuario, x veces en entrenamiento
ACAV_ROWS = 1_000_000         # negativos de ACAV100M (3 GB): con todos, la RAM de 12 GB no alcanza
VALIDATION_SHARE = 0.5        # de validation_set_features.npy (11,3 h -> ~5,7 h)
SEED = 2026

# Evaluación
THRESHOLDS = [0.5, 0.7, 0.9]
REFRACTORY_FRAMES = 25        # 2 s entre activaciones (tramos de 80 ms), como core/voice/wake.py

## 1. Instalación (versiones fijas y parches)

In [ ]:
import subprocess, sys
print(sys.version)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

In [ ]:
%cd /content
!git clone -q https://github.com/dscripka/openwakeword
!git clone -q https://github.com/dscripka/piper-sample-generator

# torch fijo para todo lo que se instala después (torch_audiomentations 0.11 no va con torchaudio 2.11).
!printf "torch==2.8.0\ntorchaudio==2.8.0\ntorchvision==0.23.0\n" > /content/torch.txt
!pip install -q -c /content/torch.txt torch==2.8.0 torchaudio==2.8.0 torchvision==0.23.0
# --no-deps: speexdsp-ns no existe para Python 3.13.
!pip install -q --no-deps -e ./openwakeword
!pip install -q -c /content/torch.txt onnxruntime ai-edge-litert onnx "datasets>=3,<4" webrtcvad \
    mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 speechbrain==0.5.14 audiomentations==0.33.0 \
    torch-audiomentations==0.11.0 acoustics==0.2.6 pronouncing==0.2.0 deep-phonemizer==0.0.19 \
    pyyaml tqdm scipy scikit-learn

In [ ]:
# Modelos de características de openWakeWord, versión v0.5.1.
!mkdir -p ./openwakeword/openwakeword/resources/models
for name in ["embedding_model.onnx", "melspectrogram.onnx"]:
    !wget -q -O ./openwakeword/openwakeword/resources/models/{name} https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/{name}
!ls -la ./openwakeword/openwakeword/resources/models

In [ ]:
# Parches. torch_audiomentations: set_audio_backend ya no existe en torchaudio 2.8.
import torch_audiomentations, pathlib
ta_dir = pathlib.Path(torch_audiomentations.__file__).parent
!grep -rl "set_audio_backend" {ta_dir} | xargs -r sed -i 's/^\(\s*\)\(torchaudio.set_audio_backend\)/\1# \2/'
!grep -rn "set_audio_backend" {ta_dir} || true

TRAIN = "/content/openwakeword/openwakeword/train.py"
# Sin generación de clips (los clips vienen en los zip): no importar piper.
!sed -i 's/^\(\s*\)from generate_samples import generate_samples/\1generate_samples = None/' {TRAIN}
# Guardar los pesos del mejor modelo antes de exportarlo (por si falla la exportación).
!sed -i 's/^\(\s*\)oww.export_model(model=best_model/\1torch.save(best_model.state_dict(), os.path.join(config["output_dir"], config["model_name"] + "_best.pt"))\n\1oww.export_model(model=best_model/' {TRAIN}
# auto_train tiene fijas las 11,3 h de la validación completa: con la mitad, contaría la mitad de falsas por hora.
VAL_HOURS = round(11.3 * VALIDATION_SHARE, 2)
!sed -i 's/val_set_hrs = 11.3$/val_set_hrs = {VAL_HOURS}/' {TRAIN}

# Sin TensorFlow no hay conversión a tflite: Asia usa el .onnx.
!sed -i 's/^\(\s*\)convert_onnx_to_tflite(/\1if False: convert_onnx_to_tflite(/' {TRAIN}

checks = {"generate_samples = None": 1, "_best.pt": 1, f"val_set_hrs = {VAL_HOURS}": 1, "if False: convert_onnx_to_tflite(": 1}
text = pathlib.Path(TRAIN).read_text()
for needle, minimum in checks.items():
    assert text.count(needle) >= minimum, f"el parche no se aplicó: {needle}"
print("Parches aplicados.")

## 2. Datos de fondo y negativos

In [ ]:
import os, numpy as np, scipy.io.wavfile, datasets
from pathlib import Path
from tqdm import tqdm

# Respuestas de sala (MIT RIR).
os.makedirs("mit_rirs", exist_ok=True)
if not os.listdir("mit_rirs"):
    rirs = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
    for row in tqdm(rirs):
        name = row["audio"]["path"].split("/")[-1]
        scipy.io.wavfile.write(os.path.join("mit_rirs", name), 16000, (row["audio"]["array"] * 32767).astype(np.int16))
print("RIR:", len(os.listdir("mit_rirs")))

In [ ]:
# AudioSet bal_train09, desde un commit fijo (el repositorio movió sus archivos). Sin FMA (el streaming de zip no va).
os.makedirs("audioset", exist_ok=True)
if not os.path.exists("audioset/bal_train09.tar"):
    !wget -q -O audioset/bal_train09.tar https://huggingface.co/datasets/agkphysics/AudioSet/resolve/4e17ae0/data/bal_train09.tar
    !cd audioset && tar -xf bal_train09.tar
os.makedirs("audioset_16k", exist_ok=True)
if not os.listdir("audioset_16k"):
    files = [str(p) for p in Path("audioset").glob("**/*.flac")]
    audioset = datasets.Dataset.from_dict({"audio": files}).cast_column("audio", datasets.Audio(sampling_rate=16000))
    for row in tqdm(audioset):
        name = row["audio"]["path"].split("/")[-1].replace(".flac", ".wav")
        scipy.io.wavfile.write(os.path.join("audioset_16k", name), 16000, (row["audio"]["array"] * 32767).astype(np.int16))
print("AudioSet:", len(os.listdir("audioset_16k")))

In [ ]:
# Negativos (ACAV100M) y validación, recortados con dd: un .npy es una cabecera y los datos seguidos.
import numpy.lib.format as npformat

def header_info(path):
    with open(path, "rb") as source:
        version = npformat.read_magic(source)
        read = npformat.read_array_header_1_0 if version == (1, 0) else npformat.read_array_header_2_0
        shape, fortran, dtype = read(source)
        return shape, fortran, dtype, source.tell()

def trim_npy(source, target, rows):
    """target con las primeras rows filas de source; borra source."""
    shape, fortran, dtype, offset = header_info(source)
    rows = min(rows, shape[0])
    new_shape = (rows,) + tuple(shape[1:])
    with open(target, "wb") as out:
        npformat.write_array_header_1_0(out, {"descr": npformat.dtype_to_descr(dtype), "fortran_order": fortran, "shape": new_shape})
    size = rows * int(np.prod(shape[1:])) * dtype.itemsize
    subprocess.run(["dd", f"if={source}", f"of={target}", "bs=4M", "iflag=skip_bytes,count_bytes", f"skip={offset}",
                    f"count={size}", "oflag=append", "conv=notrunc", "status=none"], check=True)
    os.remove(source)
    check = np.load(target, mmap_mode="r")
    print(target, check.shape, check.dtype, f"{os.path.getsize(target) / 1e9:.2f} GB")

BASE = "https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main"
if not os.path.exists("acav100m.npy"):
    !wget -q -O acav_full.npy {BASE}/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
    trim_npy("acav_full.npy", "acav100m.npy", ACAV_ROWS)
if not os.path.exists("validation.npy"):
    !wget -q -O validation_full.npy {BASE}/validation_set_features.npy
    full_rows = header_info("validation_full.npy")[0][0]
    trim_npy("validation_full.npy", "validation.npy", int(full_rows * VALIDATION_SHARE))
VALIDATION_HOURS = np.load("validation.npy", mmap_mode="r").shape[0] * 0.08 / 3600
print(f"Validación: {VALIDATION_HOURS:.2f} h")
!df -h /content | tail -1

## 3. Clips de Piper y del usuario

In [ ]:
import random, shutil, zipfile
from google.colab import files

needed = {"sintetico.zip", "mi_voz.zip"}
missing = needed - set(os.listdir("."))
if missing:
    print("Sube:", ", ".join(sorted(missing)))
    files.upload()
assert needed <= set(os.listdir(".")), f"faltan {needed - set(os.listdir('.'))}"

for name in needed:
    shutil.rmtree(name[:-4], ignore_errors=True)
    with zipfile.ZipFile(name) as archive:
        archive.extractall(name[:-4])

OUT = Path("model_output") / MODEL_NAME
for folder in ["positive_train", "positive_test", "negative_train", "negative_test"]:
    shutil.rmtree(OUT / folder, ignore_errors=True)
    (OUT / folder).mkdir(parents=True)

rng = random.Random(SEED)

def split(paths, share):
    paths = sorted(paths)
    rng.shuffle(paths)
    cut = max(1, round(len(paths) * share)) if paths else 0
    return paths[cut:], paths[:cut]

def place(paths, folder, prefix, repeat=1):
    for path in paths:
        for copy in range(repeat):
            shutil.copy(path, OUT / folder / f"{prefix}_{copy}_{Path(path).parent.name}_{Path(path).name}")

synthetic = {kind: list(Path("sintetico", kind).glob("*.wav")) for kind in ["positivos", "negativos"]}
mine = {"positivos": list(Path("mi_voz", "positivos").glob("*.wav")), "negativos": list(Path("mi_voz", "negativos").glob("*/*.wav"))}
for kind, label in [("positivos", "positive"), ("negativos", "negative")]:
    train, test = split(synthetic[kind], TEST_SHARE_SYNTHETIC)
    place(train, f"{label}_train", "piper"); place(test, f"{label}_test", "piper")
    train, test = split(mine[kind], TEST_SHARE_MINE)
    place(train, f"{label}_train", "mio", repeat=MINE_REPEAT); place(test, f"{label}_test", "mio")

print({kind: len(paths) for kind, paths in synthetic.items()}, "de Piper;", {kind: len(paths) for kind, paths in mine.items()}, "del usuario")
for folder in ["positive_train", "positive_test", "negative_train", "negative_test"]:
    print(folder, len(os.listdir(OUT / folder)))

## 4. Configuración del entrenamiento

In [ ]:
import yaml
config = yaml.safe_load(open("openwakeword/examples/custom_model.yml"))
config.update({
    "model_name": MODEL_NAME,
    "target_phrase": [TARGET_PHRASE],
    "custom_negative_phrases": [],
    "n_samples": len(os.listdir(OUT / "positive_train")),
    "n_samples_val": len(os.listdir(OUT / "positive_test")),
    "piper_sample_generator_path": "/content/piper-sample-generator",
    "output_dir": str(OUT.parent.resolve()),
    "rir_paths": [str(Path("mit_rirs").resolve())],
    "background_paths": [str(Path("audioset_16k").resolve())],
    "background_paths_duplication_rate": [1],
    "false_positive_validation_data_path": str(Path("validation.npy").resolve()),
    "feature_data_files": {"ACAV100M_sample": str(Path("acav100m.npy").resolve())},
    "augmentation_rounds": AUGMENTATION_ROUNDS,
    "steps": STEPS,
    "max_negative_weight": MAX_NEGATIVE_WEIGHT,
    "target_false_positives_per_hour": TARGET_FP_PER_HOUR,
    "layer_size": LAYER_SIZE,
})
yaml.safe_dump(config, open("my_model.yaml", "w"), allow_unicode=True)
print(yaml.safe_dump(config, allow_unicode=True))

## 5. Aumentar y entrenar (en segundo plano)

In [ ]:
# nohup: si el navegador se desconecta un momento, el entrenamiento sigue. done.txt guarda el código de salida.
!rm -f done.txt train.log
!nohup bash -c "python openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips && python openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model; echo \$? > done.txt" > train.log 2>&1 &
print("En marcha: train.log")

In [ ]:
# Progreso y memoria cada minuto, hasta que termine (con "Ejecutar todo", la evaluación espera aquí).
import time, re
from IPython.display import clear_output

def memory():
    info = dict(re.findall(r"(\w+):\s+(\d+) kB", open("/proc/meminfo").read()))
    used = (int(info["MemTotal"]) - int(info["MemAvailable"])) / 1e6
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
    return f"RAM {used:.1f} de {int(info['MemTotal']) / 1e6:.1f} GB; GPU {gpu}"

started, peak = time.time(), 0.0
while not os.path.exists("done.txt"):
    clear_output(wait=True)
    lines = open("train.log", errors="replace").read().replace("\r", "\n").splitlines() if os.path.exists("train.log") else []
    peak = max(peak, float(memory().split()[1]))
    print(f"{(time.time() - started) / 60:.0f} min · {memory()} · pico RAM {peak:.1f} GB")
    print("\n".join(lines[-15:]))
    time.sleep(60)

print("Código de salida:", open("done.txt").read().strip())
!tail -n 30 train.log
ONNX = OUT.parent / f"{MODEL_NAME}.onnx"
assert ONNX.exists(), "no se exportó el modelo: mira train.log"

## 6. Evaluación

In [ ]:
# Como la evaluación del 2026-11-05: el ONNX tiene batch fijo de 1, así que se pasa muestra a muestra.
import onnxruntime as ort
session = ort.InferenceSession(str(ONNX), providers=["CPUExecutionProvider"])
inp = session.get_inputs()[0]
frames = inp.shape[1]
print("Entrada:", inp.shape, f"{os.path.getsize(ONNX) / 1000:.0f} KB")

def score(window):
    return float(session.run(None, {inp.name: window[None].astype(np.float32)})[0].ravel()[0])

def share_over(path, threshold):
    data = np.load(path, mmap_mode="r")
    scores = [score(np.asarray(sample)[-frames:]) for sample in tqdm(data, desc=Path(path).name)]
    return scores

positive = share_over(OUT / "positive_features_test.npy", 0.5)
negative = share_over(OUT / "negative_features_test.npy", 0.5)
for threshold in THRESHOLDS:
    print(f"umbral {threshold}: positivos de prueba {np.mean(np.array(positive) >= threshold) * 100:.0f} %, "
          f"negativos de prueba {np.mean(np.array(negative) >= threshold) * 100:.1f} %")

In [ ]:
# Falsas activaciones por hora en la validación (ventanas de `frames` tramos, de una en una).
validation = np.load("validation.npy", mmap_mode="r")
scores = np.array([score(np.asarray(validation[i:i + frames])) for i in tqdm(range(0, validation.shape[0] - frames), desc="validación")])
hours = validation.shape[0] * 0.08 / 3600
for threshold in THRESHOLDS:
    count, last = 0, -REFRACTORY_FRAMES
    for index in np.flatnonzero(scores >= threshold):
        if index - last >= REFRACTORY_FRAMES:
            count, last = count + 1, index
    print(f"umbral {threshold}: {count} falsas en {hours:.1f} h = {count / hours:.2f} por hora")

## 7. Descargar el modelo

In [ ]:
# Va a data/models/openwakeword/ de Asia; se usa con voice.wake_word = MODEL_NAME en settings.json.
files.download(str(ONNX))